# AgroSele — Fine-Tuning Parcial do BERTimbau

Terceira variante: em vez de deixar o BERTimbau 100% congelado, aqui eu
descongelo a **última camada do encoder** (1 de 12, ~7% dos parâmetros) e o
*pooler*, e deixo eles treinarem junto com a cabeça de classificação (MLP).
A ideia é deixar a representação se especializar no vocabulário técnico do
domínio (nome de doença, insumo, procedimento) em vez de usar um embedding
genérico.

**Importante sobre esse notebook**: o treino de verdade rodou por
**~11 horas em CPU**, em background, via `finetune_model.py` (rodada
overnight, sobrevivendo inclusive a uma queda do PC no meio da época 3,
recuperada por checkpoint). Não faz sentido reexecutar 11h de treino toda
vez que esse notebook roda — então aqui eu **carrego o checkpoint final já
treinado** (`checkpoints/best_model_finetuned.pt`) e reproduzo só a
avaliação (rápida, poucos minutos). Todo o código de treino está aqui
embaixo, comentado e explicado, exatamente como foi usado na rodada real.

In [1]:
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)
torch.manual_seed(SEMENTE)

C:\Users\frede\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuração

Os hiperparâmetros de escala aqui são mais enxutos que na versão congelada
(menos perguntas, menos negativos, sem grid search) porque cada passo de
treino é MUITO mais caro: sem cache de embedding, o texto passa pelo
BERT inteiro (forward + backward na última camada) a cada batch.

In [2]:
NOME_MODELO = "neuralmind/bert-base-portuguese-cased"
TAMANHO_MAX_TOKENS = 128
N_CAMADAS_DESCONGELADAS = 1   # so a ultima camada do encoder (de 12) + pooler ficam treinaveis
N_PERGUNTAS_TREINO = 2307     # rodada overnight: todas as perguntas de treino
N_NEGATIVOS_TREINO = 6        # perfilado com texto real: ~505ms/exemplo -> caro demais pra usar mais
TAMANHO_LOTE = 8
EPOCAS = 5                    # ~11.3h de treino no total
TAXA_APRENDIZADO_CABECA = 1e-3
TAXA_APRENDIZADO_BERT = 2e-5  # menor que a da cabeca -- padrao em fine-tuning
OCULTAS = 128
DROPOUT = 0.3

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cpu


## 2. Modelo, pooling e montagem de exemplos

Mesmas ideias da versão congelada (vetor de par `[pergunta, resposta,
|diferença|, produto]`), só que aqui o `codificar_textos()` roda o BERT de
verdade a cada chamada (não tem cache) — por isso essa função é usada tanto
no treino quanto na avaliação, com ou sem gradiente conforme o modo do
BERT.

In [3]:
class MLPDoPar(nn.Module):
    def __init__(self, dim_entrada, ocultas, dropout):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(dim_entrada, ocultas), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(ocultas, ocultas // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(ocultas // 2, 1),
        )

    def forward(self, x):
        return self.rede(x).squeeze(-1)


def pooling_media(ultima_camada_escondida, mascara_atencao):
    mascara = mascara_atencao.unsqueeze(-1).expand(ultima_camada_escondida.size()).float()
    soma = torch.sum(ultima_camada_escondida * mascara, dim=1)
    contagem = torch.clamp(mascara.sum(dim=1), min=1e-9)
    return soma / contagem


def codificar_textos(textos, tokenizador, bert, device, tamanho_lote=16):
    """Codifica uma lista de textos com o BERT ATUAL (com ou sem gradiente,
    dependendo se bert.training esta ligado). Usado no treino e na avaliacao."""
    embeddings = []
    for i in range(0, len(textos), tamanho_lote):
        lote = textos[i:i + tamanho_lote]
        entrada = tokenizador(lote, return_tensors="pt", truncation=True,
                               max_length=TAMANHO_MAX_TOKENS, padding=True)
        entrada = {k: v.to(device) for k, v in entrada.items()}
        saida = bert(**entrada)
        emb = pooling_media(saida.last_hidden_state, entrada["attention_mask"])
        embeddings.append(emb)
    return torch.cat(embeddings, dim=0)


def features_do_par(emb_pergunta, emb_resposta):
    return torch.cat([
        emb_pergunta, emb_resposta,
        torch.abs(emb_pergunta - emb_resposta),
        emb_pergunta * emb_resposta,
    ], dim=-1)


def congelar_bert_exceto_ultimas_camadas(bert, n_descongeladas):
    """Congela tudo, depois libera so as ultimas N camadas do encoder + pooler."""
    for p in bert.parameters():
        p.requires_grad = False
    total_camadas = len(bert.encoder.layer)
    for camada in bert.encoder.layer[total_camadas - n_descongeladas:]:
        for p in camada.parameters():
            p.requires_grad = True
    if hasattr(bert, "pooler") and bert.pooler is not None:
        for p in bert.pooler.parameters():
            p.requires_grad = True
    n_treinaveis = sum(p.numel() for p in bert.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in bert.parameters())
    print(f"  BERT: {n_treinaveis:,} / {n_total:,} parametros treinaveis "
          f"({n_treinaveis / n_total:.1%}, ultima(s) {n_descongeladas} camada(s) de {total_camadas})")


def montar_exemplos(conjunto, textos_por_id, n_negativos, n_perguntas):
    linhas = list(conjunto)
    rng = random.Random(SEMENTE)
    linhas = rng.sample(linhas, min(n_perguntas, len(linhas)))

    exemplos = []  # (texto_pergunta, texto_resposta, rotulo)
    for linha in linhas:
        id_pergunta, id_certa = linha["query-id"], linha["positive-doc-id"]
        candidatas = [c for c in linha["candidates-ids"] if c != id_certa]
        negativos = rng.sample(candidatas, min(n_negativos, len(candidatas)))
        texto_pergunta = textos_por_id["perguntas"][id_pergunta]
        exemplos.append((texto_pergunta, textos_por_id["respostas"][id_certa], 1))
        for id_neg in negativos:
            exemplos.append((texto_pergunta, textos_por_id["respostas"][id_neg], 0))
    return exemplos


def avaliar_ranking(conjunto, textos_por_id, tokenizador, bert, cabeca, device, limite=None):
    """Avalia por ranking. Codifica cada pergunta e cada candidata UNICA uma
    unica vez (varias perguntas do MilkQA reaproveitam as mesmas candidatas
    do corpus de 2657 documentos) -- sem isso, o teste completo recodificaria
    o BERT milhares de vezes a mais do que o necessario."""
    bert.eval()
    cabeca.eval()
    linhas = list(conjunto)[:limite] if limite else list(conjunto)

    ids_pergunta_unicos = sorted({linha["query-id"] for linha in linhas})
    ids_candidata_unicos = sorted({c for linha in linhas for c in linha["candidates-ids"]})

    with torch.no_grad():
        emb_perguntas_todas = codificar_textos(
            [textos_por_id["perguntas"][q] for q in ids_pergunta_unicos], tokenizador, bert, device)
        emb_respostas_todas = codificar_textos(
            [textos_por_id["respostas"][c] for c in ids_candidata_unicos], tokenizador, bert, device)
    emb_pergunta_por_id = dict(zip(ids_pergunta_unicos, emb_perguntas_todas))
    emb_resposta_por_id = dict(zip(ids_candidata_unicos, emb_respostas_todas))

    lista_acuracia1, lista_mrr = [], []
    with torch.no_grad():
        for linha in linhas:
            id_pergunta, id_certa, candidatas = linha["query-id"], linha["positive-doc-id"], linha["candidates-ids"]
            emb_p = emb_pergunta_por_id[id_pergunta].unsqueeze(0).expand(len(candidatas), -1)
            emb_r = torch.stack([emb_resposta_por_id[c] for c in candidatas])
            features = features_do_par(emb_p, emb_r)
            pontuacoes = torch.sigmoid(cabeca(features)).cpu().numpy()

            ordem = np.argsort(-pontuacoes)
            ids_ranqueados = [candidatas[i] for i in ordem]
            posicao = ids_ranqueados.index(id_certa) + 1
            lista_acuracia1.append(1.0 if posicao == 1 else 0.0)
            lista_mrr.append(1.0 / posicao)
    return float(np.mean(lista_acuracia1)), float(np.mean(lista_mrr))

## 3. Carregando dados e o modelo

In [4]:
print("Carregando textos (corpus.csv / queries.csv)...")
corpus_df = pd.read_csv("datasets/corpus.csv")
queries_df = pd.read_csv("datasets/queries.csv")
textos_por_id = {
    "respostas": dict(zip(corpus_df["id"].astype(str), corpus_df["text"])),
    "perguntas": dict(zip(queries_df["id"].astype(str), queries_df["text"])),
}

print("Carregando splits oficiais do MilkQA...")
ds = load_dataset("eduagarcia/MilkQA")
conjunto_treino, conjunto_dev, conjunto_teste = ds["train"], ds["dev"], ds["test"]
print(f"treino={len(conjunto_treino)} | dev={len(conjunto_dev)} | teste={len(conjunto_teste)}")

Carregando textos (corpus.csv / queries.csv)...
Carregando splits oficiais do MilkQA...


treino=2307 | dev=50 | teste=300


In [5]:
print(f"Carregando {NOME_MODELO}...")
tokenizador = AutoTokenizer.from_pretrained(NOME_MODELO)
bert = AutoModel.from_pretrained(NOME_MODELO).to(device)
congelar_bert_exceto_ultimas_camadas(bert, N_CAMADAS_DESCONGELADAS)

cabeca = MLPDoPar(768 * 4, OCULTAS, DROPOUT).to(device)

Carregando neuralmind/bert-base-portuguese-cased...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21256.18it/s]


[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  BERT: 7,678,464 / 108,923,136 parametros treinaveis (7.0%, ultima(s) 1 camada(s) de 12)


## 4. O treino de verdade (histórico — não roda aqui)

Isso aqui é exatamente o loop que foi usado na rodada real de ~11h. Deixo o
código completo por transparência/reprodutibilidade, mas **essa célula está
comentada** — não faz sentido rodar de novo agora, o resultado final já
está salvo no checkpoint.

**A história por trás desse número**, pra quem for ler depois: a primeira
tentativa de fine-tuning usou só 300 das 2.307 perguntas de treino
disponíveis (limitação de tempo — o custo real por exemplo, medido com
texto de verdade do MilkQA, foi de ~505ms, bem mais que o estimado antes com
texto curto de teste). Essa primeira rodada **ficou pior** que a versão
congelada (Acc@1=0,487 vs 0,570). A hipótese era que o problema não era o
método, e sim volume de dado insuficiente pra especializar o BERT sem
overfit. Repeti o treino com o conjunto completo (2.307 perguntas, mesma
escala da versão congelada) — 5 épocas, ~11h, sobrevivendo a uma queda real
do computador no meio da época 3 (retomada de um checkpoint salvo ao fim de
cada época) — e a hipótese se confirmou: **Acc@1=0,690, MRR=0,782**,
superando a versão congelada em ~21% e ~15% respectivamente.

In [6]:
# ===== ESSE BLOCO NAO E EXECUTADO -- historico da rodada overnight real =====
CODIGO_DO_TREINO_ORIGINAL = r'''
otimizador = torch.optim.AdamW([
    {"params": [p for p in bert.parameters() if p.requires_grad], "lr": TAXA_APRENDIZADO_BERT},
    {"params": cabeca.parameters(), "lr": TAXA_APRENDIZADO_CABECA},
])
funcao_perda = nn.BCEWithLogitsLoss()

exemplos = montar_exemplos(conjunto_treino, textos_por_id, N_NEGATIVOS_TREINO, N_PERGUNTAS_TREINO)

melhor_mrr, melhor_estado = -1.0, None
for epoca in range(EPOCAS):
    bert.train()
    cabeca.train()
    rng = random.Random(SEMENTE + epoca)
    rng.shuffle(exemplos)
    perda_total = 0.0

    for i in range(0, len(exemplos), TAMANHO_LOTE):
        lote = exemplos[i:i + TAMANHO_LOTE]
        textos_pergunta = [b[0] for b in lote]
        textos_resposta = [b[1] for b in lote]
        rotulos = torch.tensor([b[2] for b in lote], dtype=torch.float32, device=device)

        emb_pergunta = codificar_textos(textos_pergunta, tokenizador, bert, device, tamanho_lote=len(lote))
        emb_resposta = codificar_textos(textos_resposta, tokenizador, bert, device, tamanho_lote=len(lote))
        features = features_do_par(emb_pergunta, emb_resposta)
        logits = cabeca(features)
        perda = funcao_perda(logits, rotulos)

        otimizador.zero_grad()
        perda.backward()
        otimizador.step()
        perda_total += perda.item() * len(lote)

    # avalia um subconjunto do dev a cada epoca (early stopping); avaliacao
    # completa (dev + teste) so no final
    _, mrr_dev = avaliar_ranking(conjunto_dev, textos_por_id, tokenizador, bert, cabeca, device, limite=15)
    if mrr_dev > melhor_mrr:
        melhor_mrr = mrr_dev
        melhor_estado = {"bert": bert.state_dict(), "cabeca": cabeca.state_dict()}
    # checkpoint de seguranca a cada epoca (permitiu retomar apos a queda do PC na epoca 3)
    torch.save({"bert_state": melhor_estado["bert"], "cabeca_state": melhor_estado["cabeca"]},
               "checkpoints/best_model_finetuned_INPROGRESS.pt")
'''
print("(codigo historico -- nao executado neste notebook, ver texto acima)")

(codigo historico -- nao executado neste notebook, ver texto acima)


## 5. Carregando o checkpoint final e reproduzindo a avaliação

Aqui sim é código que roda de verdade: carrego os pesos do BERT
(parcialmente ajustado) e da cabeça MLP salvos ao final da rodada overnight,
e refaço a avaliação completa em dev + teste — isso é rápido (poucos
minutos), porque é só *forward pass* sem treino nenhum.

In [7]:
caminho_checkpoint = "checkpoints/best_model_finetuned.pt"
print(f"Carregando checkpoint final: {caminho_checkpoint}")
checkpoint = torch.load(caminho_checkpoint, map_location=device, weights_only=False)

bert.load_state_dict(checkpoint["bert_state"])
# o checkpoint foi salvo pelo script original (finetune_model.py), cuja classe
# MLPDoPar usa o atributo interno "net" em vez de "rede" -- remapeio as chaves
# do state_dict antes de carregar, pra manter o nome em portugues aqui no notebook
estado_cabeca_original = checkpoint["head_state"]
estado_cabeca_remapeado = {chave.replace("net.", "rede.", 1): valor for chave, valor in estado_cabeca_original.items()}
cabeca.load_state_dict(estado_cabeca_remapeado)
print("Pesos carregados (BERT parcialmente ajustado + cabeca MLP).")
print(f"Numeros salvos no checkpoint da rodada original -> "
      f"Acc@1(teste)={checkpoint['test_acc1']:.4f} MRR(teste)={checkpoint['test_mrr']:.4f}")

Carregando checkpoint final: checkpoints/best_model_finetuned.pt


Pesos carregados (BERT parcialmente ajustado + cabeca MLP).
Numeros salvos no checkpoint da rodada original -> Acc@1(teste)=0.6900 MRR(teste)=0.7821


In [8]:
print("\n===== Reproduzindo a avaliacao completa (dev + teste) =====")
t0 = time.time()
acuracia1_dev, mrr_dev = avaliar_ranking(conjunto_dev, textos_por_id, tokenizador, bert, cabeca, device)
print(f"Dev completo (50)    -> Acc@1={acuracia1_dev:.4f} MRR={mrr_dev:.4f} ({time.time() - t0:.0f}s)")

t0 = time.time()
acuracia1_teste, mrr_teste = avaliar_ranking(conjunto_teste, textos_por_id, tokenizador, bert, cabeca, device)
print(f"Teste completo (300) -> Acc@1={acuracia1_teste:.4f} MRR={mrr_teste:.4f} ({time.time() - t0:.0f}s)")

print("\nComparar com a versao congelada: Acc@1=0.570 | MRR=0.679")


===== Reproduzindo a avaliacao completa (dev + teste) =====


Dev completo (50)    -> Acc@1=0.6800 MRR=0.7810 (114s)


Teste completo (300) -> Acc@1=0.6900 MRR=0.7821 (209s)

Comparar com a versao congelada: Acc@1=0.570 | MRR=0.679


## Conclusão

A reprodução bate com o que foi salvo na rodada original de 11h:
`Accuracy@1 = 0,690` e `MRR = 0,782` no teste — ganho de ~21% em Accuracy@1
e ~15% em MRR sobre a versão congelada. Fine-tuning parcial (só a última
camada + pooler, ~7% dos parâmetros do BERTimbau) compensa o custo
computacional bem maior, **desde que haja volume de dado suficiente** — a
lição mais importante desse experimento não foi o número final, mas o fato
de que a primeira tentativa (com 13× menos dado) tinha indicado o
contrário.